# Step 3 - Feature Engineering

Lags, rolling aggregates, weather-adjusted pour indicators, inventory turnover. Includes the leakage audit.

In [1]:
import sys
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

# Add src folder to Python path
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project Root:", PROJECT_ROOT)

Project Root: c:\Users\Olami\OneDrive\Documents\DATASCIENCEPROJECT\MIG_Cement_Demand_Forecasting


In [2]:
import pandas as pd
import numpy as np

from mig_cement.config import settings
from mig_cement.data import load, validate, preprocess


## Load Clean Data

In [3]:
panel = pd.read_parquet(settings.interim_dir / "operations_clean.parquet")

print(f"Dataset Shape: {panel.shape}")

panel.head()

Dataset Shape: (32880, 22)


,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,...,behavior,received_tonnes,rejected_delivery_tonnes,served_tonnes,induced_shortfall,was_constrained,unmet_tonnes,silo_utilisation,cover_days,y
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,...,aggressive,45.83,0.0,34.54,0,1,8.64,0.142522,1.521714,34.54
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,...,aggressive,19.97,0.0,45.26,0,0,0.00,0.086071,1.410738,45.26
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,...,aggressive,47.19,0.0,38.69,0,0,0.00,0.105045,0.996640,38.69
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,...,aggressive,18.74,0.0,33.16,0,0,0.00,0.072857,1.419180,33.16
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,...,aggressive,14.40,0.0,47.04,0,1,9.84,0.000000,0.693878,47.04


## Convert the date column and sort the data

In [12]:
# Convert date column to datetime
panel["date"] = pd.to_datetime(panel["date"])

# Sort the data by site and date
panel = panel.sort_values(["site_id", "date"]).reset_index(drop=True)

# Check the first few rows
panel.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,rain_mm,avg_temp_c,silo_capacity,...,week_of_year,available_cement,storage_utilisation,consumed_lag_1,consumed_lag_7,year,week,day,quarter,is_weekend
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,3.40,-3.10,448,...,52,98.39,0.117321,NaN,NaN,2022,52,1,1,1
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,3.23,14.28,448,...,52,83.82,0.142522,34.54,NaN,2022,52,2,1,1
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,2.64,6.40,448,...,1,85.75,0.086071,45.26,NaN,2022,1,3,1,0
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,8.25,14.23,448,...,1,65.80,0.105045,38.69,NaN,2022,1,4,1,0
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,2.69,8.97,448,...,1,47.04,0.072857,33.16,NaN,2022,1,5,1,0


## Create Calendar Features

In [13]:
panel["date"] = pd.to_datetime(panel["date"])

panel["year"] = panel["date"].dt.year
panel["month"] = panel["date"].dt.month
panel["week"] = panel["date"].dt.isocalendar().week.astype(int)
panel["day"] = panel["date"].dt.day
panel["day_of_week"] = panel["date"].dt.dayofweek
panel["quarter"] = panel["date"].dt.quarter
panel["is_weekend"] = panel["day_of_week"].isin([5,6]).astype(int)    

panel.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,rain_mm,avg_temp_c,silo_capacity,...,week_of_year,available_cement,storage_utilisation,consumed_lag_1,consumed_lag_7,year,week,day,quarter,is_weekend
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,3.40,-3.10,448,...,52,98.39,0.117321,NaN,NaN,2022,52,1,1,1
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,3.23,14.28,448,...,52,83.82,0.142522,34.54,NaN,2022,52,2,1,1
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,2.64,6.40,448,...,1,85.75,0.086071,45.26,NaN,2022,1,3,1,0
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,8.25,14.23,448,...,1,65.80,0.105045,38.69,NaN,2022,1,4,1,0
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,2.69,8.97,448,...,1,47.04,0.072857,33.16,NaN,2022,1,5,1,0


## Business Feature Engineering

In [14]:
panel["available_cement"] = (
    panel["opening_inventory_tonnes"] +
    panel["deliveries_tonnes"])


panel["storage_utilisation"] = (
    panel["opening_inventory_tonnes"] /
    panel["silo_capacity"])

panel["inventory_buffer"] = (
    panel["available_cement"] -
    panel["planned_pour_tonnes"])


panel.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,rain_mm,avg_temp_c,silo_capacity,...,available_cement,storage_utilisation,consumed_lag_1,consumed_lag_7,year,week,day,quarter,is_weekend,inventory_buffer
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,3.40,-3.10,448,...,98.39,0.117321,NaN,NaN,2022,52,1,1,1,55.21
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,3.23,14.28,448,...,83.82,0.142522,34.54,NaN,2022,52,2,1,1,38.56
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,2.64,6.40,448,...,85.75,0.086071,45.26,NaN,2022,1,3,1,0,47.06
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,8.25,14.23,448,...,65.80,0.105045,38.69,NaN,2022,1,4,1,0,32.64
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,2.69,8.97,448,...,47.04,0.072857,33.16,NaN,2022,1,5,1,0,-9.84


## Lag Features

In [15]:
# Ensure data is sorted by site and date before creating lag features
panel = panel.sort_values(["site_id", "date"]).reset_index(drop=True)

# Previous day's cement consumption
panel["consumed_lag_1"] = (
    panel.groupby("site_id")["consumed_tonnes"]
         .shift(1)
)

# Cement consumption from one week earlier
panel["consumed_lag_7"] = (
    panel.groupby("site_id")["consumed_tonnes"]
         .shift(7)
)

# Display the first 10 rows
panel.head(10)

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,rain_mm,avg_temp_c,silo_capacity,...,available_cement,storage_utilisation,consumed_lag_1,consumed_lag_7,year,week,day,quarter,is_weekend,inventory_buffer
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,3.40,-3.10,448,...,98.39,0.117321,NaN,NaN,2022,52,1,1,1,55.21
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,3.23,14.28,448,...,83.82,0.142522,34.54,NaN,2022,52,2,1,1,38.56
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,2.64,6.40,448,...,85.75,0.086071,45.26,NaN,2022,1,3,1,0,47.06
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,8.25,14.23,448,...,65.80,0.105045,38.69,NaN,2022,1,4,1,0,32.64
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,2.69,8.97,448,...,47.04,0.072857,33.16,NaN,2022,1,5,1,0,-9.84
5,2022-01-06,SITE_001,CEM_II,30.02,22.19,0.00,22.19,1.29,14.61,448,...,22.19,0.000000,47.04,NaN,2022,1,6,1,0,-7.83
6,2022-01-07,SITE_001,CEM_I,33.60,33.60,0.00,47.72,0.90,4.48,448,...,47.72,0.000000,22.19,NaN,2022,1,7,1,0,14.12
7,2022-01-08,SITE_001,CEM_I,42.12,34.28,14.12,20.16,1.95,25.92,448,...,34.28,0.031518,33.60,34.54,2022,1,8,1,1,-7.84
8,2022-01-09,SITE_001,CEM_I,0.00,0.00,0.00,34.38,1.42,16.81,448,...,34.38,0.000000,34.28,45.26,2022,1,9,1,1,34.38
9,2022-01-10,SITE_001,CEM_II,30.99,30.99,34.38,35.37,3.49,16.57,448,...,69.75,0.076741,0.00,38.69,2022,2,10,1,0,38.76


### Average Consumption

In [17]:
# 7-day rolling average consumption
panel["rolling_mean_7"] = (
    panel.groupby("site_id")["consumed_tonnes"]
         .transform(lambda x: x.shift(1).rolling(7).mean())
)

# 7-day rolling standard deviation
panel["rolling_std_7"] = (
    panel.groupby("site_id")["consumed_tonnes"]
         .transform(lambda x: x.shift(1).rolling(7).std())
)

panel.head(10)

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,rain_mm,avg_temp_c,silo_capacity,...,consumed_lag_1,consumed_lag_7,year,week,day,quarter,is_weekend,inventory_buffer,rolling_mean_7,rolling_std_7
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,3.40,-3.10,448,...,NaN,NaN,2022,52,1,1,1,55.21,NaN,NaN
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,3.23,14.28,448,...,34.54,NaN,2022,52,2,1,1,38.56,NaN,NaN
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,2.64,6.40,448,...,45.26,NaN,2022,1,3,1,0,47.06,NaN,NaN
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,8.25,14.23,448,...,38.69,NaN,2022,1,4,1,0,32.64,NaN,NaN
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,2.69,8.97,448,...,33.16,NaN,2022,1,5,1,0,-9.84,NaN,NaN
5,2022-01-06,SITE_001,CEM_II,30.02,22.19,0.00,22.19,1.29,14.61,448,...,47.04,NaN,2022,1,6,1,0,-7.83,NaN,NaN
6,2022-01-07,SITE_001,CEM_I,33.60,33.60,0.00,47.72,0.90,4.48,448,...,22.19,NaN,2022,1,7,1,0,14.12,NaN,NaN
7,2022-01-08,SITE_001,CEM_I,42.12,34.28,14.12,20.16,1.95,25.92,448,...,33.60,34.54,2022,1,8,1,1,-7.84,36.354286,8.373171
8,2022-01-09,SITE_001,CEM_I,0.00,0.00,0.00,34.38,1.42,16.81,448,...,34.28,45.26,2022,1,9,1,1,34.38,36.317143,8.383131
9,2022-01-10,SITE_001,CEM_II,30.99,30.99,34.38,35.37,3.49,16.57,448,...,0.00,38.69,2022,2,10,1,0,38.76,29.851429,15.099577


In [18]:
print(settings.processed_dir)

C:\Users\Olami\OneDrive\Documents\DATASCIENCEPROJECT\MIG_Cement_Demand_Forecasting\DATA\processed


In [19]:
# Save the engineered dataset
panel.to_parquet(
    settings.processed_dir / "operations_feature_engineered.parquet",
    index=False
)

print("✅ Engineered dataset saved successfully!")

✅ Engineered dataset saved successfully!
